# OVITO viewer for the adhesion workflow

Shows one stage's trajectory (`runs/09_pull` or `runs/05_compress`) with OVITO in the browser.
Run it from `workflows/adhesion` with a Python that has `ovito`, `ipywidgets` and `mdtraj`
(`pip install ovito jupyterlab ipywidgets`), started as `jupyter lab`.

* Topology (types, charges, bonds): `final.data` of that stage.
* Positions per frame: `trajectory.dcd`, converted once by `dcd_to_ovito.py` to
  `trajectory_ovito.dump` (`id x y z`, cell 0..L).
* Add or change modifiers in cell 3, then re-run cells 3 and 4.


In [ ]:
# 1. Which stage to show (relative to workflows/adhesion, or an absolute path)
RUN_DIR = 'projects/pmma/runs/09_pull'


In [ ]:
# 2. DCD -> OVITO dump (skipped when the dump is newer than the DCD)
from pathlib import Path
from dcd_to_ovito import convert

d = Path(RUN_DIR)
dump = convert(d)
print(f'using {dump}')


In [ ]:
# 3. The OVITO pipeline: data file + trajectory + modifiers
import numpy as np
from ovito.io import import_file
from ovito.modifiers import LoadTrajectoryModifier, SliceModifier

pipeline = import_file(str(d / 'final.data'), atom_style='full')
traj = LoadTrajectoryModifier()
traj.source.load(str(dump))
pipeline.modifiers.append(traj)

# Types are force-field names (sc4, osi, c3, ...): colour and size them by element, from the mass.
# Silica types (IFF: sc4, oc23, oc24, hoy) are drawn a little paler than the polymer.
SILICA = {'sc4', 'oc23', 'oc24', 'hoy'}
ELEMENT = [(1.2, 'H', (0.95, 0.95, 0.95), 0.30), (13.0, 'C', (0.35, 0.35, 0.35), 0.55),
           (14.5, 'N', (0.20, 0.30, 0.90), 0.55), (17.0, 'O', (0.90, 0.15, 0.10), 0.55),
           (29.0, 'Si', (0.95, 0.75, 0.25), 0.75)]
masses = {}
for line in (d / 'final.data').read_text().split('Masses', 1)[1].strip().splitlines():
    w = line.split()
    if not w or not w[0].isdigit():
        break
    masses[int(w[0])] = float(w[1])

def style_types(frame, data):
    types = data.particles_.particle_types_
    for t in types.types:
        m = masses.get(t.id, 12.0)
        for limit, el, color, radius in ELEMENT:
            if m < limit:
                break
        c = np.array(color)
        if t.name in SILICA:
            c = 0.55 * c + 0.45
        tt = types.make_mutable(t)
        tt.color, tt.radius = tuple(c), radius
pipeline.modifiers.append(style_types)

# Pulled atoms (09 only), shown in blue.
pulled_file = d / 'pulled_atoms.txt'
if pulled_file.exists():
    pulled = np.loadtxt(pulled_file, dtype=int, ndmin=1)            # 0-based
    def mark_pulled(frame, data):
        col = data.particles_.create_property('Color')
        base = np.array([t.color for t in data.particles.particle_types.types])
        idx = {t.id: k for k, t in enumerate(data.particles.particle_types.types)}
        ty = np.asarray(data.particles.particle_types)
        col[...] = base[[idx[x] for x in ty]]
        col[pulled] = (0.25, 0.45, 1.0)
    pipeline.modifiers.append(mark_pulled)

# Example: a 20-A thick slab through the middle of the cell, to see inside the film.
# pipeline.modifiers.append(SliceModifier(normal=(0, 1, 0), distance=20.0, slab_width=20.0))

print('frames:', pipeline.num_frames)


In [ ]:
# 4. Interactive viewer (drag to rotate, wheel to zoom) and a frame slider
import ipywidgets as w
from IPython.display import display
import ovito
from ovito.vis import Viewport

for p in list(ovito.scene.pipelines):
    p.remove_from_scene()
pipeline.add_to_scene()
ovito.scene.anim.last_frame = pipeline.num_frames - 1

vp = Viewport(type=Viewport.Type.Perspective, camera_dir=(0.3, 1.0, -0.25))
vp.zoom_all()
view = ovito.gui.create_ipywidget(vp, layout=w.Layout(width='100%', height='650px'))
slider = w.IntSlider(min=0, max=pipeline.num_frames - 1, value=0, description='frame',
                     continuous_update=False, layout=w.Layout(width='100%'))
slider.observe(lambda ch: setattr(ovito.scene.anim, 'current_frame', ch['new']), names='value')
display(w.VBox([view, slider]))
